In [15]:
from pyspark.sql.functions import col, when, coalesce, lit, lpad, concat_ws, to_date
from pyspark.sql.functions import col, count, when, isnan, trim

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 18, Finished, Available, Finished, False)

In [8]:
df_bronze_flights = spark.read.table("dbo.bronze_flights")
df_bronze_airlines = spark.read.table("dbo.bronze_airlines")
df_bronze_airports = spark.read.table("dbo.bronze_airports")
df_bronze_cancellation_codes = spark.read.table("dbo.bronze_cancellation_codes")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 11, Finished, Available, Finished, False)

#### Silver Layer
- Handling Nulls and duplicates 

In [4]:
df_bronze_flights.head()

StatementMeta(, cd00445b-32b3-4390-85c1-3e9454f0cac1, 7, Finished, Available, Finished, False)

Row(YEAR=2015, MONTH=1, DAY=1, DAY_OF_WEEK=4, AIRLINE='AA', FLIGHT_NUMBER=903, TAIL_NUMBER='N017AA', ORIGIN_AIRPORT='EGE', DESTINATION_AIRPORT='DFW', SCHEDULED_DEPARTURE=1225, DEPARTURE_TIME=1300, DEPARTURE_DELAY=35, TAXI_OUT=15, WHEELS_OFF=1315, SCHEDULED_TIME=130, ELAPSED_TIME=137, AIR_TIME=110, DISTANCE=721, WHEELS_ON=1605, TAXI_IN=12, SCHEDULED_ARRIVAL=1535, ARRIVAL_TIME=1617, ARRIVAL_DELAY=42, DIVERTED=0, CANCELLED=0, CANCELLATION_REASON='', AIR_SYSTEM_DELAY=7, SECURITY_DELAY=19, AIRLINE_DELAY=5, LATE_AIRCRAFT_DELAY=11, WEATHER_DELAY=0)

In [7]:
null_counts = df_bronze_flights.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) 
    for c in df_bronze_flights.columns
])

display(null_counts)

StatementMeta(, cd00445b-32b3-4390-85c1-3e9454f0cac1, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b698c2d4-52c8-4022-a75c-ceb59a152119)

In [10]:
total_rows_flights = df_bronze_flights.count()
print(f"Fights: {total_rows_flights:,}\n" + "="*50)

total_rows_airlines= df_bronze_airlines.count()
print(f"Airlines: {total_rows_airlines:,}\n" + "="*50)

total_rows_airports = df_bronze_airports.count()
print(f"Airports: {total_rows_airports:,}\n" + "="*50)

total_rows_cancellation_codes = df_bronze_cancellation_codes.count()
print(f"Cancellation Codes: {total_rows_cancellation_codes:,}\n" + "="*50)

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 13, Finished, Available, Finished, False)

Fights: 5,819,079
Airlines: 14
Airports: 322
Cancellation Codes: 4


In [16]:
# Checking for nulls
for c in df_bronze_flights.columns:
    # Count nulls and NaNs for the current column
    null_count = df_bronze_flights.filter(col(c).isNull() | isnan(col(c))).count()
    
    # Calculate percentage
    null_pct = (null_count / total_rows) * 100
    
    # Print formatted output
    print(f"Column '{c}': {null_count:,} nulls ({null_pct:.2f}%)")

StatementMeta(, cd00445b-32b3-4390-85c1-3e9454f0cac1, 34, Finished, Available, Finished, False)

Column 'YEAR': 0 nulls (0.00%)
Column 'MONTH': 0 nulls (0.00%)
Column 'DAY': 0 nulls (0.00%)
Column 'DAY_OF_WEEK': 0 nulls (0.00%)
Column 'AIRLINE': 0 nulls (0.00%)
Column 'FLIGHT_NUMBER': 0 nulls (0.00%)
Column 'TAIL_NUMBER': 0 nulls (0.00%)
Column 'ORIGIN_AIRPORT': 0 nulls (0.00%)
Column 'DESTINATION_AIRPORT': 0 nulls (0.00%)
Column 'SCHEDULED_DEPARTURE': 0 nulls (0.00%)
Column 'DEPARTURE_TIME': 86,153 nulls (1.48%)
Column 'DEPARTURE_DELAY': 86,153 nulls (1.48%)
Column 'TAXI_OUT': 89,047 nulls (1.53%)
Column 'WHEELS_OFF': 89,047 nulls (1.53%)
Column 'SCHEDULED_TIME': 6 nulls (0.00%)
Column 'ELAPSED_TIME': 105,071 nulls (1.81%)
Column 'AIR_TIME': 105,071 nulls (1.81%)
Column 'DISTANCE': 0 nulls (0.00%)
Column 'WHEELS_ON': 92,513 nulls (1.59%)
Column 'TAXI_IN': 92,513 nulls (1.59%)
Column 'SCHEDULED_ARRIVAL': 0 nulls (0.00%)
Column 'ARRIVAL_TIME': 92,513 nulls (1.59%)
Column 'ARRIVAL_DELAY': 105,071 nulls (1.81%)
Column 'DIVERTED': 0 nulls (0.00%)
Column 'CANCELLED': 0 nulls (0.00%)
Co

In [17]:
# 2. Silver Transformation & Null Cleanup
df_silver_flights = df_bronze_flights \
    .withColumn("DEP_DELAY", coalesce(col("DEPARTURE_DELAY").cast("integer"), lit(0))) \
    .withColumn("ARR_DELAY", coalesce(col("ARRIVAL_DELAY").cast("integer"), lit(0))) \
    .withColumn("TAXI_OUT", coalesce(col("TAXI_OUT").cast("integer"), lit(0))) \
    .withColumn("TAXI_IN", coalesce(col("TAXI_IN").cast("integer"), lit(0))) \
    .withColumn("AIR_TIME", coalesce(col("AIR_TIME").cast("integer"), lit(0))) \
    .withColumn("ELAPSED_TIME", coalesce(col("ELAPSED_TIME").cast("integer"), lit(0))) \
    .withColumn("DISTANCE", col("DISTANCE").cast("integer")) \
    .withColumn("IS_DELAYED", when(col("DEPARTURE_DELAY") > 15, 1).otherwise(0)) \
    .withColumn("IS_CANCELLED", col("CANCELLED").cast("integer")) \
    .withColumn("IS_DIVERTED", col("DIVERTED").cast("integer")) \
    .withColumn("AIR_SYSTEM_DELAY", coalesce(col("AIR_SYSTEM_DELAY").cast("integer"), lit(0))) \
    .withColumn("SECURITY_DELAY", coalesce(col("SECURITY_DELAY").cast("integer"), lit(0))) \
    .withColumn("AIRLINE_DELAY", coalesce(col("AIRLINE_DELAY").cast("integer"), lit(0))) \
    .withColumn("LATE_AIRCRAFT_DELAY", coalesce(col("LATE_AIRCRAFT_DELAY").cast("integer"), lit(0))) \
    .withColumn("WEATHER_DELAY", coalesce(col("WEATHER_DELAY").cast("integer"), lit(0))) \
    .withColumn("CANCELLATION_REASON", coalesce(col("CANCELLATION_REASON"), lit("N/A")))

# 3. Deduplicate (Safety check for primary key uniqueness)
df_silver_flights = df_silver_flights.dropDuplicates([
    "YEAR", "MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE"
])

# 4. Save to Silver Layer Delta Table
df_silver_flights.write.format("delta").mode("overwrite").saveAsTable("dbo.silver_flights")

print("dbo.silver_flights table successfully created and written!")

StatementMeta(, cd00445b-32b3-4390-85c1-3e9454f0cac1, 35, Finished, Available, Finished, False)

dbo.silver_flights table successfully created and written!


In [11]:
# Checking for nulls
for c in df_bronze_airlines.columns:
    # Count nulls and NaNs for the current column
    null_count = df_bronze_airlines.filter(col(c).isNull() | isnan(col(c))).count()
    
    # Calculate percentage
    null_pct = (null_count / total_rows_airlines) * 100
    
    # Print formatted output
    print(f"Column '{c}': {null_count:,} nulls ({null_pct:.2f}%)")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 14, Finished, Available, Finished, False)

Column 'IATA_CODE': 0 nulls (0.00%)
Column 'AIRLINE': 0 nulls (0.00%)


In [16]:
# Silver Processing: Clean whitespace and deduplicate
df_silver_airlines = df_bronze_airlines \
    .withColumn("AIRLINE_CODE", trim(col("IATA_CODE"))) \
    .withColumn("AIRLINE_NAME", trim(col("AIRLINE"))) \
    .filter(col("AIRLINE_CODE").isNotNull()) \
    .dropDuplicates(["AIRLINE_CODE"])

# Save to Silver Delta Table
df_silver_airlines.write.format("delta").mode("overwrite").saveAsTable("dbo.silver_airlines")

print("dbo.silver_airlines processed and written to Silver layer successfully!")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 19, Finished, Available, Finished, False)

dbo.silver_airlines processed and written to Silver layer successfully!


In [12]:
for c in df_bronze_airports.columns:
    # Count nulls and NaNs for the current column
    null_count = df_bronze_airports.filter(col(c).isNull() | isnan(col(c))).count()
    
    # Calculate percentage
    null_pct = (null_count / total_rows_airports) * 100
    
    # Print formatted output
    print(f"Column '{c}': {null_count:,} nulls ({null_pct:.2f}%)")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 15, Finished, Available, Finished, False)

Column 'IATA_CODE': 0 nulls (0.00%)
Column 'AIRPORT': 0 nulls (0.00%)
Column 'CITY': 0 nulls (0.00%)
Column 'STATE': 0 nulls (0.00%)
Column 'COUNTRY': 0 nulls (0.00%)
Column 'LATITUDE': 3 nulls (0.93%)
Column 'LONGITUDE': 3 nulls (0.93%)


In [13]:
# Silver Transformation: Handle missing coordinates and clean string fields
df_silver_airports = df_bronze_airports \
    .withColumn("AIRPORT_CODE", col("IATA_CODE")) \
    .withColumn("LATITUDE", coalesce(col("LATITUDE").cast("double"), lit(0.0))) \
    .withColumn("LONGITUDE", coalesce(col("LONGITUDE").cast("double"), lit(0.0)))

# Deduplicate by Airport Code
df_silver_airports = df_silver_airports.dropDuplicates(["AIRPORT_CODE"])

# Write to Silver Delta Table
df_silver_airports.write.format("delta").mode("overwrite").saveAsTable("dbo.silver_airports")

print("dbo.silver_airports processed and saved successfully!")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 16, Finished, Available, Finished, False)

dbo.silver_airports processed and saved successfully!


In [17]:
for c in df_bronze_cancellation_codes.columns:
    # Count nulls and NaNs for the current column
    null_count = df_bronze_cancellation_codes.filter(col(c).isNull() | isnan(col(c))).count()
    
    # Calculate percentage
    null_pct = (null_count / total_rows_cancellation_codes) * 100
    
    # Print formatted output
    print(f"Column '{c}': {null_count:,} nulls ({null_pct:.2f}%)")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 20, Finished, Available, Finished, False)

Column 'CANCELLATION_REASON': 0 nulls (0.00%)
Column 'CANCELLATION_DESCRIPTION': 0 nulls (0.00%)


In [19]:
df_silver_canc = df_bronze_cancellation_codes \
    .withColumn("REASON_CODE", trim(col("CANCELLATION_REASON"))) \
    .withColumn("REASON_DESCRIPTION", trim(col("CANCELLATION_DESCRIPTION"))) \
    .filter(col("REASON_CODE").isNotNull()) \
    .dropDuplicates(["REASON_CODE"])

# Save to Silver Delta Table
df_silver_canc.write.format("delta").mode("overwrite").saveAsTable("dbo.silver_cancellation_codes")

print("dbo.silver_cancellation_codes table written to Silver layer successfully!")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 22, Finished, Available, Finished, False)

dbo.silver_cancellation_codes table written to Silver layer successfully!


#### Gold Layer

In [20]:
# ==============================================================================
# GOLD LAYER: STAR SCHEMA FOR POWER BI DIRECT LAKE
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. GOLD DIMENSION: Airlines (gold_dim_airlines)
# ------------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TABLE dbo.gold_dim_airlines AS
    SELECT DISTINCT 
        AIRLINE_CODE AS airline_code,
        AIRLINE_NAME AS airline_name
    FROM dbo.silver_airlines
""")

# ------------------------------------------------------------------------------
# 2. GOLD DIMENSION: Airports (gold_dim_airports)
# ------------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TABLE dbo.gold_dim_airports AS
    SELECT DISTINCT 
        AIRPORT_CODE AS airport_code,
        AIRPORT AS airport_name,
        CITY AS city,
        STATE AS state,
        COUNTRY AS country,
        LATITUDE AS latitude,
        LONGITUDE AS longitude
    FROM dbo.silver_airports
""")

# ------------------------------------------------------------------------------
# 3. GOLD DIMENSION: Cancellation Codes (gold_dim_cancellation_codes)
# ------------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TABLE dbo.gold_dim_cancellation_codes AS
    SELECT DISTINCT 
        REASON_CODE AS cancellation_code,
        REASON_DESCRIPTION AS cancellation_description
    FROM dbo.silver_cancellation_codes
""")

# ------------------------------------------------------------------------------
# 4. GOLD FACT: Flight Operations (gold_fact_flights)
# ------------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TABLE dbo.gold_fact_flights AS
    SELECT 
        -- Fact Key / Identifiers
        f.YEAR AS year,
        f.MONTH AS month,
        f.DAY AS day,
        f.DAY_OF_WEEK AS day_of_week,
        f.FLIGHT_NUMBER AS flight_number,
        f.TAIL_NUMBER AS tail_number,
        
        -- Foreign Keys to Dimension Tables
        f.AIRLINE AS airline_code,
        f.ORIGIN_AIRPORT AS origin_airport_code,
        f.DESTINATION_AIRPORT AS dest_airport_code,
        f.CANCELLATION_REASON AS cancellation_code,

        -- Operational Timings & Delays (Measures)
        f.SCHEDULED_DEPARTURE AS scheduled_departure,
        f.DEPARTURE_TIME AS departure_time,
        f.DEP_DELAY AS departure_delay_mins,
        f.TAXI_OUT AS taxi_out_mins,
        f.WHEELS_OFF AS wheels_off_time,
        f.SCHEDULED_TIME AS scheduled_time_mins,
        f.ELAPSED_TIME AS elapsed_time_mins,
        f.AIR_TIME AS air_time_mins,
        f.DISTANCE AS distance_miles,
        f.WHEELS_ON AS wheels_on_time,
        f.TAXI_IN AS taxi_in_mins,
        f.SCHEDULED_ARRIVAL AS scheduled_arrival,
        f.ARRIVAL_TIME AS arrival_time,
        f.ARR_DELAY AS arrival_delay_mins,

        -- Delay Breakdown Categories (Measures)
        f.AIR_SYSTEM_DELAY AS air_system_delay_mins,
        f.SECURITY_DELAY AS security_delay_mins,
        f.AIRLINE_DELAY AS airline_delay_mins,
        f.LATE_AIRCRAFT_DELAY AS late_aircraft_delay_mins,
        f.WEATHER_DELAY AS weather_delay_mins,

        -- Binary Status Flags
        f.IS_DELAYED AS is_delayed_flag,
        f.IS_CANCELLED AS is_cancelled_flag,
        f.IS_DIVERTED AS is_diverted_flag,

        -- Metadata / Lineage Audit Column
        'BATCH_DATAFLOW_GEN2' AS record_source,
        CURRENT_TIMESTAMP() AS gold_ingestion_timestamp
    FROM dbo.silver_flights f
""")

print("Gold Star Schema successfully created! Ready for Power BI Direct Lake.")

StatementMeta(, 4cce4517-2584-4b84-bcc2-437f061ee75b, 23, Finished, Available, Finished, False)

Gold Star Schema successfully created! Ready for Power BI Direct Lake.
